# Nik Studio - animate

**You give it pictures and a song. It gives you one finished MP4.**

### Check first. It is free.

Colab charges for a GPU from the moment it connects, whatever you run on
it. So do the checking with no GPU at all:

1. **Runtime > Change runtime type > CPU**
2. Run **cell 2**. It finds your files, checks every one of them, and
   stops with `this cost nothing`.
3. Only once it says that: **Runtime > Change runtime type > L4 GPU**,
   run **cell 1**, then **cell 2** again.

A CPU runtime costs nothing, so a wrong folder or a picture that will
not open is found for free instead of on a meter.

Pick **L4**, not A100. The model is small and A100 spends units far
faster for no better result.

### Before you start

Make this folder in Google Drive and put your files in it:

```
My Drive / NikStudio / Input /
      Scene01.png
      Scene02.png
      Scene03.png
      song.mp3
```

Any names work - the pictures are used **in order**, so number them, and
any one audio file is taken as the song. `python tools\prepare.py --copy`
builds this folder for you and checks it before you ever open Colab.

The finished video comes back as:

```
My Drive / NikStudio / Output / Episode.mp4
```

### What it does

Every picture becomes a moving clip, the clips are cut to share the
length of the song exactly, and the song is laid over the top.

It reads the card and picks its own quality settings, so there is
nothing to tune.

It saves each clip to Drive as it finishes. If the session dies, run it
again - it picks up where it stopped instead of starting over. A picture
you replace is noticed and made again; the rest are not.

**Honest about one thing:** the mouth moves, but it is not lip-synced to
the words. Nothing free does real lip-sync yet.


In [ ]:
# ======================================================================
# CELL 1 of 2 - the packages.  Only needed once a GPU is turned on
# ======================================================================
#
# Skip this while you are still checking on a CPU runtime - cell 2 does
# the checking with what Colab already has.
#
# If Colab offers "RESTART SESSION" when this finishes, click it.
# Cell 2 depends on nothing in here, so a restart costs nothing.

!pip install -q "diffusers>=0.32" "transformers>=4.44" accelerate safetensors sentencepiece bitsandbytes imageio-ffmpeg

print("Packages installed. Now run cell 2.")


In [ ]:
# ======================================================================
# CELL 2 of 2 - the whole thing
# ======================================================================
#
# Finds your pictures and your song in Drive, animates every picture,
# cuts the clips to share the song's length, lays the song over the top,
# and plays you the result.
#
# It depends on nothing above it, so running cells out of order or
# letting Colab restart the runtime cannot break it.

import gc
import json
import re
import subprocess
import time

from pathlib import Path

import torch

from PIL import Image


# ======================================================================
# SETTINGS - the only part worth editing
# ======================================================================

DRIVE = "/content/drive/MyDrive"

FOLDER = DRIVE + "/NikStudio"

# What HAPPENS in the shot. Not what the picture shows - that is already
# there. One clear physical action beats three vague ones.
# One action for the character - and then a plain statement that
# everything else stays put.
#
# Whatever is not named drifts. Naming only the boy left the puppy, the
# kitten and the duckling free to melt, and they did, from about a
# second and a half in. And dropping "the camera does not move" let the
# whole frame push slowly in on its own.
#
# So: one movement, then who else is in the shot, then the camera.
PROMPT = (
    "The little boy sways gently from side to side, smiling. "
    "The puppy, the kitten and the duckling stay where they are, "
    "watching him. The background does not change. "
    "The camera does not move."
)

# A picture can have its own, by file name. Anything not listed here
# uses PROMPT above.
PROMPTS = {
    # "Scene02.png": "The boy splashes the water with both hands, ...",
}

# "static image, no movement" used to be in here, and it was the single
# worst line in this notebook: it tells the model to look UNLIKE the
# still it was given, which is the one thing it must not do. The face
# melted at two seconds and the scene was gone by four.
#
# Ask for the faults you do not want. Do not ask it to leave the picture.
NEGATIVE = (
    "worst quality, blurry, distorted, deformed face, melting face, "
    "morphing, warping, extra limbs, watermark, text, "
    "camera zoom, camera pan, changing background, characters vanishing"
)

# Seconds per picture when there is no song to divide up.
FALLBACK_SECONDS = 5.0

FPS = 24

# Which model. Left blank, the machine decides: the 13B on a card with
# room for it, the 2B otherwise. Set it to "big" or "small" to insist.
FORCE_MODEL = ""

# How long one generated clip is, before looping.
#
# This is the number that decides whether the video looks right, and
# shorter is safer. Everything in the picture drifts as the clip goes
# on, and the smallest things go first: over three seconds the
# butterflies had melted by one second and the animals' faces by about
# one and a half. Two seconds stays clean.
#
# A picture that has to be on screen longer is not given a longer clip.
# The clip is played forwards, then backwards, then forwards again, for
# as long as it is needed - a sway reads as continuous that way, and the
# model is never asked for more than it can do.
#
# Raise it for more movement in one go, at the cost of more drifting.
# The honest fix for a long song is more pictures, not longer clips.
CLIP_SECONDS = 2.0

# Make ONE clip from the first picture and stop.
#
# Worth doing before every real run, and certainly before a new
# character or a new prompt: two minutes of GPU tells you what the model
# does with your picture, instead of finding out eleven clips later.
# The song is ignored while this is on.
TEST_ONE_PICTURE = False


# The run stops when there are too few pictures for the song, because
# the clips would have to be slowed past the point of looking right
# and a paid GPU should not be spent finding that out. Set this True
# to go ahead anyway.
ALLOW_SLOW_CLIPS = False



# ======================================================================
# The card decides the quality, not you
# ======================================================================

# Not having one is fine here. Colab charges for a GPU from the moment
# it connects, whatever you run on it, so everything that does not need
# one is done first - on a CPU runtime, which costs nothing. Only when
# the files are known to be right is a GPU worth connecting.
HAS_GPU = torch.cuda.is_available()

if HAS_GPU:
    CARD = torch.cuda.get_device_name(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    # Assume the card the settings would be chosen for, so the advice
    # about picture counts is the advice you will actually get.
    CARD, VRAM = "none yet - checking your files first", 24.0

# Compute capability 8.0 (Ampere) or newer is where bfloat16 is real.
# Do not ask torch.cuda.is_bf16_supported() - it says True on a T4,
# because torch emulates bfloat16 in software rather than refusing, and
# that emulation is slower than it is worth.
MAJOR = torch.cuda.get_device_capability()[0] if HAS_GPU else 8

DTYPE = torch.bfloat16 if MAJOR >= 8 else torch.float16

# Ordinary RAM matters as much as the card here, and is the thing that
# actually killed the earlier attempts. The 9GB text encoder is unpacked
# in RAM before it ever reaches the GPU, so a big card on a small-RAM
# runtime still dies. Squeeze the text encoder to 8-bit whenever either
# one is short, not just when the card is.
try:
    import psutil

    RAM = psutil.virtual_memory().total / 1e9

except ImportError:
    # Colab ships psutil, but a notebook that dies on a missing helper
    # before it has even looked at your files is no use to anyone.
    import os

    RAM = (
        os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    )

ROOMY_CARD = VRAM >= 20
ROOMY_RAM = RAM >= 20

# What the card decides is whether the 9GB text encoder has to be
# squeezed into 8-bit. It does NOT decide the video size: that is set by
# what the model was trained on, not by how much room there is.
QUANTISE = not (ROOMY_CARD and ROOMY_RAM)

# ---------------------------------------------------------- the model

# LTX comes in two sizes and the difference is not subtle.
#
# The 2B is the original. It runs anywhere and it drifts - the smallest
# things in a picture melt within a second or two.
#
# The 13B is distilled, which means it does in 8 steps what the 2B needs
# 50 for, and it takes image_cond_noise_scale - a setting the 2B's
# pipeline does not even have. At 0.0 no noise is added to your picture
# before it starts, which is precisely the thing that lets a clip wander
# away from it. It needs a 24GB card and room in ordinary RAM to be
# streamed through.
BIG = (VRAM >= 20 and RAM >= 30) if not FORCE_MODEL else FORCE_MODEL == "big"

if BIG:
    MODEL = "Lightricks/LTX-Video-0.9.8-13B-distilled"

    # Distilled: few steps, no classifier-free guidance, and the exact
    # schedule the model was distilled for. These are not tuneable -
    # they came with the model.
    STEPS = 8
    TIMESTEPS = [1000, 993, 987, 981, 975, 909, 725, 0.03]
    GUIDANCE = 1.0

    WIDTH, HEIGHT = 960, 544

else:
    MODEL = "Lightricks/LTX-Video"

    STEPS = 50
    TIMESTEPS = None
    GUIDANCE = 3.0

    # This one was trained near 704x480 - about 338,000 pixels. Asking
    # for 1024x576 is 75% more, and it showed: the picture came apart
    # after two seconds. 768x448 is what it knows, in 16:9.
    WIDTH, HEIGHT = 768, 448

# The finished video is scaled up to this. Upscaling a clip that held
# together beats generating one that did not.
OUTPUT_WIDTH, OUTPUT_HEIGHT = 1280, 720

# Frames the model is asked for. (frames - 1) has to divide by 8, and
# the rounding goes to the nearest - rounding down turned a 3.0s clip
# into a 2.7s one.
MAX_FRAMES = max(25, round((CLIP_SECONDS * FPS - 1) / 8) * 8 + 1)

print(f"GPU         : {CARD}" + (f" ({VRAM:.0f}GB)" if HAS_GPU else ""))
print(f"System RAM  : {RAM:.0f}GB")
print(f"Precision   : {'bfloat16' if MAJOR >= 8 else 'float16'}"
      f"{', text encoder in 8-bit' if QUANTISE else ''}")
print(f"Model       : {MODEL.split('/')[-1]}"
      f" ({'13B' if BIG else '2B'}, {STEPS} steps)")
print(f"Generated at: {WIDTH}x{HEIGHT}, "
      f"{MAX_FRAMES} frames ({MAX_FRAMES / FPS:.1f}s)")
print(f"Video out   : {OUTPUT_WIDTH}x{OUTPUT_HEIGHT}")


# ======================================================================
# Your files
# ======================================================================

try:
    from google.colab import drive

except ImportError:
    drive = None        # not in Colab; the folder is expected to be there

if drive is not None and not Path(DRIVE).exists():

    try:
        drive.mount("/content/drive")

    except Exception as trouble:

        # "credential propagation was unsuccessful" is the usual one, and
        # it is the browser, not Drive and not this notebook. Three lines
        # of plain English beat three frames of traceback.
        raise SystemExit(
            f"Google Drive would not connect.\n\n"
            f"    {type(trouble).__name__}: {trouble}\n\n"
            "This is the browser refusing the sign-in popup, not a "
            "problem with your files.\n"
            "In order of what usually works:\n\n"
            "  1. Allow third-party cookies for colab.research.google.com, "
            "then run this cell\n     again. In Chrome: the eye or padlock "
            "icon in the address bar > Cookies.\n"
            "  2. Use an ordinary window, not Incognito or a guest "
            "profile.\n"
            "  3. Runtime > Disconnect and delete runtime, reconnect, and "
            "try once more.\n"
            "  4. If a popup appeared and you closed it, just run the cell "
            "again and accept it.\n\n"
            "Nothing has been generated, so nothing is lost."
        ) from trouble

root = Path(FOLDER)

INPUT = root / "Input"
OUTPUT = root / "Output"
CLIPS = OUTPUT / "Clips"

def looks_like_input(folder):
    """A folder with pictures in it is a folder worth offering."""

    try:
        return any(
            item.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp")
            for item in folder.iterdir()
        )
    except OSError:
        return False


def hunt_for_input():
    """
    Where the pictures actually are.

    "No such folder" is a dead end when the folder plainly exists on the
    other machine - Drive may be a different account, or the path may be
    a level off. Looking is more use than complaining, so anything named
    Input with pictures in it counts, and so does a NikStudio folder.
    """

    drive = Path(DRIVE)

    if not drive.exists():
        return []

    found = []

    for depth in ("*", "*/*", "*/*/*", "*/*/*/*"):

        for folder in drive.glob(depth):

            if not folder.is_dir():
                continue

            if folder.name.lower() in ("input", "nikstudio"):
                if looks_like_input(folder):
                    found.append(folder)

    return sorted(set(found))


if not INPUT.exists():

    print(f"\n  {INPUT} is not there. Looking for it ...")

    candidates = hunt_for_input()

    if len(candidates) == 1:

        INPUT = candidates[0]

        root = INPUT.parent
        OUTPUT = root / "Output"
        CLIPS = OUTPUT / "Clips"

        print(f"  Found your pictures in {INPUT} - using that.")

    elif candidates:

        raise SystemExit(
            f"No such folder: {INPUT}\n\n"
            "These have pictures in them - put the right one in FOLDER "
            "at the top of this cell\n(FOLDER is the folder ABOVE Input):"
            "\n\n"
            + "\n".join(f"    {folder}" for folder in candidates)
        )

    else:

        visible = sorted(
            item.name for item in Path(DRIVE).glob("*") if item.is_dir()
        ) if Path(DRIVE).exists() else []

        raise SystemExit(
            f"No such folder: {INPUT}\n\n"
            "Nothing with pictures in it was found anywhere in this "
            "Drive.\n\n"
            "Two things to check:\n"
            "  1. Colab is mounted on the same Google account your Drive "
            "folder is on.\n"
            "  2. Google Drive on your PC has finished uploading - a "
            "folder that only\n     exists locally is not there yet.\n\n"
            "The top level of the Drive Colab can see:\n\n"
            + ("\n".join(f"    {name}" for name in visible[:30])
               or "    (nothing)")
        )

CLIPS.mkdir(parents=True, exist_ok=True)

def natural_key(path):
    """
    Sort "Scene2" before "Scene10".

    Plain alphabetical order puts "10" before "2", which silently
    shuffles someone's scenes. Numbers in a name are compared as numbers.
    """

    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", path.name)
    ]


PICTURES = sorted(
    (path for path in INPUT.iterdir()
     if path.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp")),
    key=natural_key,
)

if not PICTURES:
    raise SystemExit(f"No pictures in {INPUT}.")

SONG = next(
    (
        path for path in sorted(INPUT.iterdir(), key=natural_key)
        if path.suffix.lower() in (".mp3", ".wav", ".m4a", ".aac", ".ogg")
    ),
    None,
)


def seconds_of(media):
    """How long an audio or video file runs, in seconds."""

    probe = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(media),
        ],
        capture_output=True,
        text=True,
    )

    try:
        return float(probe.stdout.strip())
    except ValueError:
        return 0.0


if TEST_ONE_PICTURE:

    PICTURES = PICTURES[:1]

    # Nothing is laid over a test clip. What comes back is the model's
    # work and nothing else.
    SONG = None

    SONG_SECONDS = 0.0

    # One clip, its own natural length, no looping and no song. This is
    # the model's work with nothing else laid over it.
    SHARE = MAX_FRAMES / FPS

    print(f"\nTEST       : one clip from {PICTURES[0].name}, "
          f"{SHARE:.1f}s. The song is ignored.")

elif SONG:
    SONG_SECONDS = seconds_of(SONG)
    SHARE = SONG_SECONDS / len(PICTURES)
    print(f"\nSong       : {SONG.name} ({SONG_SECONDS:.1f}s)")

else:
    SONG_SECONDS = 0.0
    SHARE = FALLBACK_SECONDS
    print("\nSong       : none found - using "
          f"{FALLBACK_SECONDS:.0f}s per picture")

print(f"Pictures   : {len(PICTURES)} "
      f"({', '.join(p.name for p in PICTURES)})")
print(f"Each holds : {SHARE:.1f}s")

# ======================================================================
# Everything that can go wrong, found before the GPU is touched
# ======================================================================
#
# Loading the model takes minutes and a paid GPU is charged for them, so
# nothing here is left to be discovered halfway through the run.

problems = []

print()

for picture_file in PICTURES:

    try:
        with Image.open(picture_file) as check:
            check.verify()

        with Image.open(picture_file) as check:
            shape = check.size

    except Exception:
        problems.append(
            f"{picture_file.name} will not open. Re-save it as a PNG."
        )
        continue

    print(f"  {picture_file.name:<28} {shape[0]}x{shape[1]}")

if SONG and not SONG_SECONDS:
    problems.append(
        f"{SONG.name} cannot be read. Try a plain MP3 or WAV."
    )

# One clip played forwards and back covers this much. Beyond a couple
# of those, the same three seconds coming round again starts to be
# noticed, however gentle the movement.
BOUNCE = (2 * MAX_FRAMES - 2) / FPS

if SHARE > BOUNCE * 4:

    enough = max(1, int(SONG_SECONDS / (BOUNCE * 2) + 0.999))

    complaint = (
        f"{len(PICTURES)} picture(s) over {SONG_SECONDS:.0f}s means each "
        f"is on screen {SHARE:.0f}s, and a clip\n"
        f"       covers {BOUNCE:.1f}s, so the same movement would repeat "
        f"{SHARE / BOUNCE:.0f} times over.\n"
        f"       Use about {enough} pictures for a song this long.\n\n"
        f"       Only trying one picture out? Set TEST_ONE_PICTURE = True "
        f"at the top - it makes\n       one clip, ignores the song, and "
        f"takes about two minutes. Or ALLOW_SLOW_CLIPS = True\n"
        f"       to go ahead as things stand."
    )

    if ALLOW_SLOW_CLIPS:
        print(f"\n  Warning: {complaint}")
    else:
        problems.append(complaint)

elif SHARE > BOUNCE * 2:
    print(f"\n  Note: {SHARE:.1f}s a picture against a {BOUNCE:.1f}s clip "
          f"means the movement comes round\n        "
          f"{SHARE / BOUNCE:.1f} times. Watchable, but more pictures would "
          f"be better.")

elif SHARE > MAX_FRAMES / FPS:
    print(f"\n  Note: each {MAX_FRAMES / FPS:.1f}s clip is played forwards "
          f"and back to fill its {SHARE:.1f}s.\n        Nothing is "
          f"stretched, so nothing judders.")

if problems:

    raise SystemExit(
        "\n\nStopping before the GPU is used:\n\n"
        + "\n".join(f"  -  {problem}" for problem in problems)
        + "\n\nFix these and run this cell again. Nothing has been "
          "charged for."
    )

print("\n  Everything checks out.")

if not HAS_GPU:

    raise SystemExit(
        "\n\nYour files are ready, and this cost nothing - there was no "
        "GPU running.\n\n"
        "Now turn one on and do the work:\n\n"
        "  1. Runtime > Change runtime type > L4 GPU > Save\n"
        "  2. Run cell 1 (the packages), then run this cell again.\n\n"
        f"It will make {len(PICTURES)} clip(s). Nothing else needs "
        "checking."
    )


# ======================================================================
# The model.  Kept between runs - loading it is most of the wait
# ======================================================================

if "pipe" not in globals():

    if BIG:
        from diffusers import LTXConditionPipeline as Pipeline
    else:
        from diffusers import LTXImageToVideoPipeline as Pipeline

    print("\nLoading the model. A few minutes the first time"
          + (" - the 13B is a big download." if BIG else "."))

    started = time.time()

    parts = {}

    if QUANTISE:

        from transformers import BitsAndBytesConfig, T5EncoderModel

        # 9GB of text encoder will not fit through a small card's RAM at
        # full size. In 8-bit it goes straight to the GPU at about 4.7GB.
        parts["text_encoder"] = T5EncoderModel.from_pretrained(
            MODEL,
            subfolder="text_encoder",
            quantization_config=BitsAndBytesConfig(load_in_8bit=True),
            device_map="auto",
        )

    pipe = Pipeline.from_pretrained(MODEL, torch_dtype=DTYPE, **parts)

    if BIG:
        # 13B in bfloat16 is 26GB and a 24GB card is a 24GB card. Parts
        # are moved on and off as they are needed - slower per step, but
        # a distilled model only takes eight of them.
        pipe.enable_model_cpu_offload()

    elif QUANTISE:
        # The text encoder is already on the card; move what is left.
        pipe.transformer.to("cuda")
        pipe.vae.to("cuda")

    else:
        pipe.to("cuda")

    # Decoding every frame in one piece is what runs a card out of
    # memory. Tiling decodes it in patches instead.
    pipe.vae.enable_tiling()

    print(f"Model ready in {time.time() - started:.0f}s "
          f"({torch.cuda.memory_allocated() / 1e9:.1f}GB on the card).")

else:
    print("\nModel already loaded - reusing it.")


# ======================================================================
# One picture -> one clip
# ======================================================================

def fitted(picture):
    """Cover the frame and crop the overflow, rather than squash."""

    scale = max(WIDTH / picture.width, HEIGHT / picture.height)

    picture = picture.resize(
        (round(picture.width * scale), round(picture.height * scale)),
        Image.LANCZOS,
    )

    left = (picture.width - WIDTH) // 2
    top = (picture.height - HEIGHT) // 2

    return picture.crop((left, top, left + WIDTH, top + HEIGHT))


def animate(picture_file, clip_file, seconds):
    """
    One picture -> one clip of exactly `seconds`.

    The model is always asked for the same short clip, however long the
    picture has to be on screen. Asking it for a long one is what made
    the face melt. The clip is then played forwards, backwards and
    forwards again until the time is filled: a sway or a clap reads as
    continuous that way, and the seam is at the moment the movement
    turns around, where it is least visible.

    Returns (frames, loops) - what was generated, and how many times the
    forward-and-back pair had to run.
    """

    image = fitted(Image.open(picture_file).convert("RGB"))

    prompt = PROMPTS.get(picture_file.name, PROMPT)

    def run(count):

        asked = dict(
            image=image,
            prompt=prompt,
            negative_prompt=NEGATIVE,
            width=WIDTH,
            height=HEIGHT,
            num_frames=count,
            frame_rate=FPS,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE,
            generator=torch.Generator("cpu").manual_seed(42),
        )

        if BIG:
            asked.update(
                timesteps=TIMESTEPS,
                # No noise on the conditioning picture. This is the
                # setting that keeps the clip on the picture it was
                # given, and the 2B pipeline has no equivalent.
                image_cond_noise_scale=0.0,
                decode_timestep=0.05,
                decode_noise_scale=0.025,
            )

        return pipe(**asked).frames[0]

    frames = MAX_FRAMES

    try:
        video = run(frames)

    except torch.cuda.OutOfMemoryError:

        gc.collect()
        torch.cuda.empty_cache()

        frames = max(25, ((frames // 2 - 1) // 8) * 8 + 1)

        print(f"    card ran out of room - shorter clip ({frames} frames)")

        video = run(frames)

    from diffusers.utils import export_to_video

    raw = clip_file.with_name(clip_file.stem + "_raw.mp4")

    export_to_video(video, str(raw), fps=FPS)

    made = frames / FPS

    # ------------------------------------------------- forwards and back

    if seconds > made:

        bounced = clip_file.with_name(clip_file.stem + "_bounce.mp4")

        # The reversed half drops a frame at each end. The first is the
        # one the forward half just finished on, and the last is the one
        # the loop is about to start on again - keep either and that
        # picture is held for two frames, which shows as a hitch.
        subprocess.run(
            [
                "ffmpeg", "-y", "-loglevel", "error",
                "-i", str(raw),
                "-filter_complex",
                "[0:v]split[fwd][back];"
                f"[back]reverse,trim=start_frame=1:end_frame={frames - 1},"
                "setpts=PTS-STARTPTS[rev];"
                "[fwd][rev]concat=n=2:v=1[out]",
                "-map", "[out]",
                "-r", str(FPS),
                "-c:v", "libx264", "-pix_fmt", "yuv420p",
                "-preset", "veryfast", "-crf", "18",
                str(bounced),
            ],
            check=True,
        )

        source, unit = bounced, (2 * frames - 2) / FPS

    else:
        source, unit = raw, made

    loops = seconds / unit if unit else 1.0

    # ------------------------------------------- fill the slot exactly

    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-stream_loop", "-1", "-i", str(source),
            "-t", f"{seconds:.3f}",
            "-vf", f"scale={OUTPUT_WIDTH}:{OUTPUT_HEIGHT}:flags=lanczos",
            "-r", str(FPS),
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-preset", "medium", "-crf", "20",
            str(clip_file),
        ],
        check=True,
    )

    for temporary in (raw, clip_file.with_name(clip_file.stem + "_bounce.mp4")):
        temporary.unlink(missing_ok=True)

    return frames, loops


# ======================================================================
# Every picture
# ======================================================================

# A clip is reused only when the thing it was made from has not moved.
# Skipping by file name alone is what would quietly leave you with a clip
# of last week's Scene02 after you replaced the picture.
STAMPS = CLIPS / "made.json"

try:
    stamps = json.loads(STAMPS.read_text(encoding="utf-8"))
except Exception:
    stamps = {}


def stamp_for(picture_file):

    facts = picture_file.stat()

    return {
        "picture": picture_file.name,
        "bytes": facts.st_size,
        "modified": int(facts.st_mtime),
        "prompt": PROMPTS.get(picture_file.name, PROMPT),
        "size": f"{WIDTH}x{HEIGHT}",
        "seconds": round(SHARE, 2),
    }


made = []

for number, picture_file in enumerate(PICTURES, start=1):

    clip_file = CLIPS / f"{picture_file.stem}.mp4"

    label = f"[{number}/{len(PICTURES)}] {picture_file.name}"

    wanted = stamp_for(picture_file)

    finished = clip_file.exists() and clip_file.stat().st_size > 0

    if finished and stamps.get(clip_file.name) == wanted:
        print(f"{label}: already made, skipping")
        made.append(clip_file)
        continue

    if finished:
        print(f"{label}: picture or prompt changed - making it again")

    print(f"{label}: animating {SHARE:.1f}s ...")

    started = time.time()

    frames, loops = animate(picture_file, clip_file, SHARE)

    if loops > 1.05:
        note = (f", {frames / FPS:.1f}s clip played back and forth "
                f"{loops:.1f}x to fill {SHARE:.1f}s")
    else:
        note = ""

    print(f"    done in {(time.time() - started) / 60:.1f} min{note}")

    # Written after every clip, not at the end: a session that dies has
    # to leave behind an honest record of what is really finished.
    stamps[clip_file.name] = wanted

    STAMPS.write_text(json.dumps(stamps, indent=1), encoding="utf-8")

    made.append(clip_file)


# ======================================================================
# Join them, lay the song over the top
# ======================================================================

print("\nJoining the clips ...")

listing = OUTPUT / "clips.txt"

listing.write_text(
    "".join(f"file '{clip}'\n" for clip in made),
    encoding="utf-8",
)

FINAL = OUTPUT / "Episode.mp4"

command = [
    "ffmpeg", "-y", "-loglevel", "error",
    "-f", "concat", "-safe", "0", "-i", str(listing),
]

if SONG:
    command += ["-i", str(SONG), "-c:a", "aac", "-b:a", "192k", "-shortest"]

command += [
    "-c:v", "libx264", "-pix_fmt", "yuv420p",
    "-preset", "medium", "-crf", "20",
    "-r", str(FPS),
    str(FINAL),
]

subprocess.run(command, check=True)

listing.unlink(missing_ok=True)

print(f"\nFinished: {FINAL}")
print(f"Length  : {seconds_of(FINAL):.1f}s")

from IPython.display import Video, display

display(Video(str(FINAL), embed=True, width=min(WIDTH, 720)))

# ----------------------------------------------------------------------
# The file is in Drive, so it is already on your PC if Drive syncs there.
#
# Want a picture to do something else? Add it to PROMPTS at the top,
# delete that clip from Output/Clips, and run this cell again - the other
# clips are kept, so only the one you changed is made afresh.
# ----------------------------------------------------------------------
